In [2]:
import sys
sys.path.append("..")

from src.database.db_connection import get_connection

conn = get_connection()
print(conn.execute("SELECT sqlite_version()").fetchone())
conn.close()

('3.50.4',)


In [ ]:
import pandas as pd
import joblib
from src.database.db_connection import get_connection

# Delay: data + model load
delay_df = pd.read_pickle("../data/processed/final_dataset_for_model_training.pkl")
delay_model = joblib.load("../models/delay_prediction_model.joblib")

# use only those features which the models wants
delay_features = list(delay_model.feature_names_in_)
delay_df["delay_prob"] = delay_model.predict_proba(delay_df[delay_features])[:, 1]
delay_df["delay_pred"] = (delay_df["delay_prob"] >= 0.5).astype(int)

#those features which are used for identification of the order and its item, shipping mode and region
id_cols = [c for c in ["order_id", "order_item_id", "shipping_mode", "order_region"] if c in delay_df.columns]
print("ID columns mile:", id_cols)

delay_out = delay_df[id_cols + ["delay_prob", "delay_pred"]]

ID columns mile: []


In [7]:
print(delay_df.shape)
print(delay_df.columns.tolist())
print(delay_df.index[:10].tolist())
print(delay_df.index.is_monotonic_increasing)

(172762, 62)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'shipping_gap', 'mode_avg_delay', 'mode_delay_std', 'region_avg_delay', 'region_delay_std', 'order_month', 'order_day_of_week', 'ord

In [2]:
import pandas as pd
check = pd.read_pickle("../data/processed/shortage_engineered_dataset.pkl")
print(check.shape)
print(check.columns.tolist())
print(check.index[:10].tolist())

(172762, 54)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode', 'shipping_gap', 'order_week', 'shortage_risk', 'product_demand_mean', 'product_demand_std', 'product_qty_mean', 'pr

In [ ]:
import pandas as pd
import joblib

shortage_df = pd.read_pickle("../data/processed/shortage_engineered_dataset.pkl")
shortage_model = joblib.load("../models/shortage_prediction_model.joblib")

# predictions (features ke naam already snake_case hain, unko chhedna nahi hai)
shortage_features = list(shortage_model.feature_names_in_)
shortage_df["shortage_prob"] = shortage_model.predict_proba(shortage_df[shortage_features])[:, 1]
shortage_df["shortage_pred"] = (shortage_df["shortage_prob"] >= 0.5).astype(int)

# column names ko orders table jaisa banao
shortage_df.columns = [c.strip().lower().replace("(", "").replace(")", "").replace(" ", "_") for c in shortage_df.columns]

id_cols_s = [c for c in ["order_id", "order_item_id", "product_card_id", "category_name"] if c in shortage_df.columns]
print("ID columns mile:", id_cols_s)

shortage_out = shortage_df[id_cols_s + ["shortage_prob", "shortage_pred"]]
print(shortage_out.head())

ID columns mile: ['order_id', 'order_item_id', 'product_card_id', 'category_name']
   order_id  order_item_id  product_card_id category_name  shortage_prob  \
0     72831         176146             1359  Pet Supplies       0.715189   
1     72832         176147             1359  Pet Supplies       0.715149   
2     72833         176148             1359  Pet Supplies       0.715109   
3     72834         176149             1359  Pet Supplies       0.715068   
4     72837         176152             1359  Pet Supplies       0.715028   

   shortage_pred  
0              1  
1              1  
2              1  
3              1  
4              1  


In [6]:
import sys
sys.path.append("..")

import pandas as pd
import joblib
from src.database.db_connection import get_connection

def snake(cols):
    return [c.strip().lower().replace("(", "").replace(")", "").replace(" ", "_") for c in cols]

# ---------- DELAY ----------
delay_df = pd.read_pickle("../data/processed/final_dataset_for_model_training.pkl")
delay_model = joblib.load("../models/delay_prediction_model.joblib")

delay_features = list(delay_model.feature_names_in_)
delay_df["delay_prob"] = delay_model.predict_proba(delay_df[delay_features])[:, 1]
delay_df["delay_pred"] = (delay_df["delay_prob"] >= 0.5).astype(int)

delay_df.columns = snake(delay_df.columns)     # predictions ke BAAD rename
id_cols_d = [c for c in ["order_id", "order_item_id", "shipping_mode", "order_region"] if c in delay_df.columns]
print("Delay ID columns:", id_cols_d)
delay_out = delay_df[id_cols_d + ["delay_prob", "delay_pred"]]

# ---------- SHORTAGE ----------
shortage_df = pd.read_pickle("../data/processed/shortage_engineered_dataset.pkl")
shortage_model = joblib.load("../models/shortage_prediction_model.joblib")

shortage_features = list(shortage_model.feature_names_in_)
shortage_df["shortage_prob"] = shortage_model.predict_proba(shortage_df[shortage_features])[:, 1]
shortage_df["shortage_pred"] = (shortage_df["shortage_prob"] >= 0.5).astype(int)

shortage_df.columns = snake(shortage_df.columns)
id_cols_s = [c for c in ["order_id", "order_item_id", "product_card_id", "category_name"] if c in shortage_df.columns]
shortage_out = shortage_df[id_cols_s + ["shortage_prob", "shortage_pred"]]

# ---------- DB mein save ----------
conn = get_connection()
delay_out.to_sql("delay_predictions", conn, if_exists="replace", index=False)
shortage_out.to_sql("shortage_predictions", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS delay_rows FROM delay_predictions", conn))
print(pd.read_sql("SELECT COUNT(*) AS shortage_rows FROM shortage_predictions", conn))
conn.close()


Delay ID columns: ['order_id', 'order_item_id', 'order_region']
   delay_rows
0      172762
   shortage_rows
0         172762


In [7]:
import pandas as pd
conn = get_connection()

# 1. Tables kaun kaun si hain
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

# 2. Shipping mode ke hisaab se delay risk (JOIN + GROUP BY)
q1 = """
SELECT o.shipping_mode,
       COUNT(*) AS total_orders,
       ROUND(AVG(d.delay_prob), 3) AS avg_delay_prob,
       ROUND(100.0 * SUM(d.delay_pred) / COUNT(*), 1) AS predicted_late_pct
FROM orders o
JOIN delay_predictions d ON o.order_item_id = d.order_item_id
GROUP BY o.shipping_mode
ORDER BY avg_delay_prob DESC
"""
print(pd.read_sql(q1, conn))

# 3. Category ke hisaab se shortage risk
q2 = """
SELECT category_name,
       COUNT(*) AS total_orders,
       ROUND(AVG(shortage_prob), 3) AS avg_shortage_prob,
       ROUND(100.0 * SUM(shortage_pred) / COUNT(*), 1) AS predicted_shortage_pct
FROM shortage_predictions
GROUP BY category_name
ORDER BY avg_shortage_prob DESC
LIMIT 10
"""
print(pd.read_sql(q2, conn))

conn.close()

                   name
0                orders
1     delay_predictions
2  shortage_predictions
    shipping_mode  total_orders  avg_delay_prob  predicted_late_pct
0     First Class         26512           1.000               100.0
1    Second Class         33806           0.801               100.0
2        Same Day          9293           0.469                23.9
3  Standard Class        103151           0.398                 0.1
        category_name  total_orders  avg_shortage_prob  predicted_shortage_pct
0   Golf Bags & Carts            61              0.743                    93.4
1    Men's Golf Clubs           274              0.739                    94.9
2              Garden           466              0.738                   100.0
3          Golf Balls          1403              0.736                    98.9
4        Boxing & MMA           402              0.736                    98.8
5         Accessories          1697              0.736                    99.1
6          

In [9]:
conn = get_connection()   # reopen the connection

q3 = """
SELECT shipping_mode,
       ROUND(100.0 * AVG(late_delivery_risk), 1) AS actual_late_pct
FROM orders
GROUP BY shipping_mode
"""
print(pd.read_sql(q3, conn))

conn.close()

    shipping_mode  actual_late_pct
0     First Class            100.0
1        Same Day             47.9
2    Second Class             79.8
3  Standard Class             39.8


In [10]:
print("Actual shortage rate:", shortage_df["shortage_risk"].mean())
print("Predicted shortage rate:", shortage_df["shortage_pred"].mean())
print(shortage_df["shortage_prob"].describe())

Actual shortage rate: 0.02918465866336347
Predicted shortage rate: 0.3179402877947697
count    1.727620e+05
mean     3.901302e-01
std      2.108919e-01
min      8.600509e-07
25%      2.184849e-01
50%      3.724964e-01
75%      5.500140e-01
max      8.727231e-01
Name: shortage_prob, dtype: float64


In [11]:
# Delay: levels based on probability (it is well calibrated)
delay_out["risk_level"] = pd.cut(
    delay_out["delay_prob"],
    bins=[-0.01, 0.45, 0.75, 1.0],
    labels=["Low", "Medium", "High"]
)

# Shortage: levels based on rank (score is not a true probability)
shortage_out["risk_level"] = pd.qcut(
    shortage_out["shortage_prob"].rank(method="first"),
    q=[0, 0.70, 0.90, 1.0],
    labels=["Low", "Medium", "High"]
)

conn = get_connection()
delay_out.assign(risk_level=delay_out["risk_level"].astype(str)).to_sql("delay_predictions", conn, if_exists="replace", index=False)
shortage_out.assign(risk_level=shortage_out["risk_level"].astype(str)).to_sql("shortage_predictions", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT risk_level, COUNT(*) AS n FROM delay_predictions GROUP BY risk_level", conn))
print(pd.read_sql("SELECT risk_level, COUNT(*) AS n FROM shortage_predictions GROUP BY risk_level", conn))
conn.close()

  risk_level       n
0       High   59950
1        Low  105845
2     Medium    6967
  risk_level       n
0       High   17277
1        Low  120933
2     Medium   34552


In [12]:
conn = get_connection()

# 1. Top 10 orders that are High risk for BOTH delay and shortage
q_top = """
SELECT d.order_id, d.order_item_id, o.shipping_mode, o.order_region, o.category_name,
       ROUND(d.delay_prob, 3)    AS delay_prob,
       ROUND(s.shortage_prob, 3) AS shortage_prob
FROM delay_predictions d
JOIN shortage_predictions s ON s.order_item_id = d.order_item_id
JOIN orders o               ON o.order_item_id = d.order_item_id
WHERE d.risk_level = 'High' AND s.risk_level = 'High'
ORDER BY d.delay_prob + s.shortage_prob DESC
LIMIT 10
"""
print(pd.read_sql(q_top, conn))

# 2. Delay risk by region (how many orders are High risk in each region)
q_region = """
SELECT order_region,
       COUNT(*)                                   AS orders,
       ROUND(AVG(delay_prob), 3)                  AS avg_delay_prob,
       ROUND(100.0 * AVG(risk_level = 'High'), 1) AS high_risk_pct
FROM delay_predictions
GROUP BY order_region
ORDER BY avg_delay_prob DESC
"""
print(pd.read_sql(q_region, conn))

# 3. How many orders are High risk on both delay and shortage
q_both = """
SELECT COUNT(*) AS both_high
FROM delay_predictions d
JOIN shortage_predictions s ON s.order_item_id = d.order_item_id
WHERE d.risk_level = 'High' AND s.risk_level = 'High'
"""
print(pd.read_sql(q_both, conn))

conn.close()

   order_id  order_item_id shipping_mode     order_region category_name  \
0     69808         173123   First Class  Northern Europe      Cameras    
1     69809         173124   First Class  Northern Europe      Cameras    
2     69816         173131   First Class   Western Europe      Cameras    
3     69819         173134   First Class   Western Europe      Cameras    
4     69820         173135   First Class   Western Europe      Cameras    
5     69821         173136   First Class   Western Europe      Cameras    
6     69831         173146   First Class   Western Europe      Cameras    
7     69849         173164   First Class  Southern Europe      Cameras    
8     69850         173165   First Class  Southern Europe      Cameras    
9     69881         173196   First Class  Northern Europe      Cameras    

   delay_prob  shortage_prob  
0         1.0          0.873  
1         1.0          0.873  
2         1.0          0.873  
3         1.0          0.873  
4         1.0      

In [1]:
import sys
sys.path.append("..")
from src.database import queries

print(queries.delay_by_mode())
print(queries.shortage_by_category(5))
print(queries.top_risky_orders(3))

    shipping_mode  orders  avg_delay_prob  high_risk_pct
0     First Class   26512           1.000          100.0
1    Second Class   33806           0.801           98.9
2        Same Day    9293           0.469            0.0
3  Standard Class  103151           0.398            0.0
       category_name  orders  avg_shortage_prob  high_risk_pct
0  Golf Bags & Carts      61              0.743           93.4
1   Men's Golf Clubs     274              0.739           87.6
2             Garden     466              0.738           59.2
3         Golf Balls    1403              0.736           73.3
4       Boxing & MMA     402              0.736           76.1
   order_id  order_item_id shipping_mode     order_region category_name  \
0     69808         173123   First Class  Northern Europe      Cameras    
1     69809         173124   First Class  Northern Europe      Cameras    
2     69816         173131   First Class   Western Europe      Cameras    

   delay_prob  shortage_prob  
0    

In [2]:
import sys
sys.path.append("..")
import pandas as pd
import joblib
from src.database.db_connection import get_connection

def snake(cols):
    return [c.strip().lower().replace("(", "").replace(")", "").replace(" ", "_") for c in cols]

# ---------- DELAY ----------
delay_df = pd.read_pickle("../data/processed/final_dataset_for_model_training.pkl")
delay_model = joblib.load("../models/delay_prediction_model.joblib")

delay_features = list(delay_model.feature_names_in_)
delay_df["delay_prob"] = delay_model.predict_proba(delay_df[delay_features])[:, 1]
delay_df["delay_pred"] = (delay_df["delay_prob"] >= 0.5).astype(int)
delay_df.columns = snake(delay_df.columns)   # rename AFTER predicting

delay_fact_cols = ["mode_avg_delay", "mode_delay_std", "mode_reliability_score",
                   "region_avg_delay", "region_delay_std", "region_reliability_score"]
id_cols_d = [c for c in ["order_id", "order_item_id", "order_region"] if c in delay_df.columns]

delay_out = delay_df[id_cols_d + delay_fact_cols + ["delay_prob", "delay_pred"]].copy()
delay_out["risk_level"] = pd.cut(delay_out["delay_prob"], bins=[-0.01, 0.45, 0.75, 1.0],
                                 labels=["Low", "Medium", "High"]).astype(str)

# ---------- SHORTAGE ----------
shortage_df = pd.read_pickle("../data/processed/shortage_engineered_dataset.pkl")
shortage_model = joblib.load("../models/shortage_prediction_model.joblib")

shortage_features = list(shortage_model.feature_names_in_)
shortage_df["shortage_prob"] = shortage_model.predict_proba(shortage_df[shortage_features])[:, 1]
shortage_df["shortage_pred"] = (shortage_df["shortage_prob"] >= 0.5).astype(int)
shortage_df.columns = snake(shortage_df.columns)

shortage_fact_cols = ["product_demand_mean", "product_demand_std",
                      "qty_deviation", "product_order_history_count"]
id_cols_s = [c for c in ["order_id", "order_item_id", "product_card_id", "category_name"] if c in shortage_df.columns]

shortage_out = shortage_df[id_cols_s + shortage_fact_cols + ["shortage_prob", "shortage_pred"]].copy()
shortage_out["risk_level"] = pd.qcut(shortage_out["shortage_prob"].rank(method="first"),
                                     q=[0, 0.70, 0.90, 1.0],
                                     labels=["Low", "Medium", "High"]).astype(str)

# ---------- Save to DB ----------
conn = get_connection()
delay_out.to_sql("delay_predictions", conn, if_exists="replace", index=False)
shortage_out.to_sql("shortage_predictions", conn, if_exists="replace", index=False)

# New table to cache LLM answers
conn.execute("""
CREATE TABLE IF NOT EXISTS llm_briefings (
    id             INTEGER PRIMARY KEY AUTOINCREMENT,
    kind           TEXT NOT NULL,      -- delay_briefing, delay_whatif, shortage_briefing, shortage_whatif
    entity_key     TEXT NOT NULL,      -- e.g. order_item_id or category name
    prompt_version TEXT NOT NULL,      -- v1, v2, v3
    question       TEXT,               -- only used for what-if questions
    response       TEXT NOT NULL,
    created_at     TEXT DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

print(pd.read_sql("SELECT * FROM delay_predictions LIMIT 3", conn).T)
print(pd.read_sql("SELECT * FROM shortage_predictions LIMIT 3", conn).T)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
conn.close()

                                        0              1              2
order_id                                1              2              2
order_item_id                           1              2              4
order_region              Central America  South America  South America
mode_avg_delay                  -0.006301      -0.006301      -0.006301
mode_delay_std                   1.418683       1.418683       1.418683
mode_reliability_score           0.565501           -2.0           -1.5
region_avg_delay                 0.557886       0.555979       0.555979
region_delay_std                 1.487178       1.509928       1.509928
region_reliability_score         0.565501       0.565501           -1.0
delay_prob                       0.420577       0.414275       0.372125
delay_pred                              0              0              0
risk_level                            Low            Low            Low
                                        0             1         

In [1]:
import sys
sys.path.append("..")
from src.database.db_connection import get_connection

conn = get_connection()
for table in ["orders", "shortage_predictions"]:
    conn.execute(f"UPDATE {table} SET category_name = TRIM(category_name)")
conn.commit()
print(conn.execute("SELECT COUNT(*) FROM orders WHERE category_name LIKE '% '").fetchone())
conn.close()

(0,)
